In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
import numpy as np
from numpy.linalg import norm
import math
import pandas as pd
from functools import reduce


def build_block_encoding_circuit(n_list, D):
    """
    Build block-encoding circuit for the D-dimensional Laplacian \tilde{L}.
    - n_list: list or tuple of integers [n_0, n_1, ..., n_{D-1}] where each n_i is number of qubits
              in index register |j_i>. Number of grid points along dimension i is 2**n_i.
    - D: logical number of dimensions (may not be a power of two)
    Circuit layout (qubit ordering):
      [ all index qubits concatenated: |j_0> ... |j_{D-1}> ]
      [ ancilla l1 ]
      [ ancilla l0 ]
      [ selector ancilla d qubits (bottom) ]
    Returns: QuantumCircuit
    """
    assert isinstance(n_list, (list, tuple)) and len(n_list) >= 1
    assert len(n_list) == D, "Length of n_list must equal D"

    # total system index qubits
    sum_n = sum(n_list)

    # d = number of ancilla selector qubits (smallest power of two >= D)
    d = math.ceil(math.log2(max(1, D)))
    D_hat = 2**d

    total_qubits = sum_n + 2 + d
    qc = QuantumCircuit(total_qubits, name=f"blk_enc_L{D}h_n{'-'.join(map(str,n_list))}")

    # index qubit ranges per register
    index_ranges = []
    start = 0
    for ni in n_list:
        index_ranges.append(list(range(start, start + ni)))
        start += ni

    # ancillas
    l1 = sum_n
    l0 = sum_n + 1
    anc_sel = list(range(sum_n + 2, total_qubits)) if d > 0 else []
    print(anc_sel)

    # --- PART 1: H then Z on each small ancilla ---
    qc.h(l1); qc.z(l1)
    qc.h(l0); qc.z(l0)

    # --- PART 1b: H on selector ancilla (H_d) to create uniform superposition ---
    for q in anc_sel:
        qc.h(q)

    # --- helper: build increment/decrement subcircuit for a register size ni ---
    def make_increment_subcircuit(ni):
        circ = QuantumCircuit(ni, name=f"INC{ni}")
        for k in range(ni):
            if k == 0:
                circ.x(0)
            else:
                circ.mcx(list(range(k)), k)
        return circ

    # Prebuild inc/dec gates for each distinct ni to reuse when possible
    inc_gates = {}
    dec_gates = {}
    for ni in set(n_list):
        inc_sub = make_increment_subcircuit(ni)
        dec_sub = inc_sub.inverse()
        inc_gates[ni] = inc_sub.to_gate()
        dec_gates[ni] = dec_sub.to_gate()

    # --- PART 2: For each dimension i, controlled S_- and S_+ when selector ancilla indicate i ---
    for i, reg_qubits in enumerate(index_ranges):
        ni = len(reg_qubits)
        inc_gate = inc_gates[ni]
        dec_gate = dec_gates[ni]

        # number of controls: 1 (l1 or l0) + d
        n_ctrls = 1 + d

        if n_ctrls > 0:
            c_inc = inc_gate.control(n_ctrls)
            c_dec = dec_gate.control(n_ctrls)
        else:
            c_inc = inc_gate
            c_dec = dec_gate

        # set ancilla pattern for selecting |i>:
        bitstr = format(i, f"0{d}b") if d > 0 else ""
        # invert ancilla bits where bit is '0' so that after inversion all ancilla bits are 1
        for k, b in enumerate(bitstr[::-1]):  # anc_sel[0] is LSB
            if b == '0':
                qc.x(anc_sel[k])

        # --- controlled S_- on l1 == 0: implement by X(l1) then control on l1 + anc_sel ---
        qc.x(l1)
        controls = [l1] + anc_sel
        qc.append(c_dec, controls + reg_qubits)
        qc.x(l1)

        # --- controlled S_+ on l0 == 1 ---
        controls = [l0] + anc_sel
        qc.append(c_inc, controls + reg_qubits)

        # undo ancilla bit inversions
        for k, b in enumerate(bitstr[::-1]):
            if b == '0':
                qc.x(anc_sel[k])

    # --- PART 3: final Hadamards on small ancillas ---
    qc.h(l1); qc.h(l0)
    # and final H_d on selector ancilla to invert the initial H_d
    for q in anc_sel:
        qc.h(q)

    print(qc)

    return qc

# -------------------------
# Classical reference Laplacian
# -------------------------
def shift_plus(N):
    S = np.zeros((N, N), dtype=complex)
    for j in range(N):
        S[(j + 1) % N, j] = 1.0
    return S

def shift_minus(N):
    S = np.zeros((N, N), dtype=complex)
    for j in range(N):
        S[(j - 1) % N, j] = 1.0
    return S

def laplacian_1d(N):
    """Scaled 1D Laplacian \tilde L on N grid points (periodic)."""
    return 0.25 * (shift_minus(N) - 2 * np.eye(N) + shift_plus(N))


def laplacian_multiD_qiskit(Ns):
    """
    Build D-dimensional Laplacian with periodic BCs,
    in the *same qubit ordering convention as Qiskit*.
    Ns = [N0, N1, ..., N_{D-1}], where Ni = 2**n_i.
    """
    D = len(Ns)
    N_total = np.prod(Ns)
    L_total = np.zeros((N_total, N_total), dtype=complex)

    for i, Ni in enumerate(Ns):
        Li = laplacian_1d(Ni)
        # Build tensor with reversed order (so last entry in Ns = qubit 0)
        kron_terms = []
        for j, Nj in enumerate(Ns):
            if j == i:
                kron_terms.append(Li)
            else:
                kron_terms.append(np.eye(Nj, dtype=complex))
        # reverse list for Qiskit little-endian convention
        kron_terms = kron_terms[::-1]
        Li_full = kron_terms[0]
        for term in kron_terms[1:]:
            Li_full = np.kron(Li_full, term)
        L_total += Li_full

    d = math.ceil(math.log2(max(1, D)))

    return L_total * (1 / (2**d))  # scale by 1/D


# -------------------------
# Test function 
# -------------------------
def test_block_encoding_general(n_list, D):
    qc = build_block_encoding_circuit(n_list, D)
    U = Operator(qc).data

    Ns = [2**ni for ni in n_list]
    N_total = int(np.prod(Ns))
    d = math.ceil(math.log2(max(1, D)))
    anc_dim = 2**(2 + d)

    U4 = U.reshape((anc_dim, N_total, anc_dim, N_total))
    top_left_block = U4[0, :, 0, :]

    # classical Laplacian with Qiskit ordering
    L_total = laplacian_multiD_qiskit(Ns)
    diff_norm = norm(top_left_block - L_total)
    print(f"||top_left_block - expected L_total|| = {diff_norm:.3e}")
    
    rng = np.random.default_rng(12345)
    v = rng.normal(size=(N_total,)) + 1j * rng.normal(size=(N_total,))
    v /= norm(v)
    e0 = np.zeros((anc_dim,), dtype=complex); e0[0] = 1.0
    psi0 = np.kron(e0, v) # we take the initial state |0...0> , if anything else then first prepare the state before applying U
    psi_out = U.dot(psi0)
    projected = psi_out[:N_total] 
    diff_vec_norm = norm(projected - L_total.dot(v))
    print(f"||projected - L_total v|| = {diff_vec_norm:.3e}")
    
    return diff_norm, L_total

# -------------------------
# Example
# -------------------------
if __name__ == "__main__":
    n_list = [2,2,3]   # j0 has 4 points, j1 has 8 points, j2 has 8 points
    D = 3 # this should match len(n_list)
    print("Testing D=2 with n_list =", n_list)
    _, L_total= test_block_encoding_general(n_list, D)

Testing D=2 with n_list = [2, 2, 3]
[9, 10]
                     ┌──────────┐     ┌───────┐                           »
 q_0: ───────────────┤0         ├─────┤0      ├───────────────────────────»
                     │  INC2_dg │     │  INC2 │                           »
 q_1: ───────────────┤1         ├─────┤1      ├───────────────────────────»
                     └────┬─────┘     └───┬───┘          ┌──────────┐     »
 q_2: ────────────────────┼───────────────┼──────────────┤0         ├─────»
                          │               │              │  INC2_dg │     »
 q_3: ────────────────────┼───────────────┼──────────────┤1         ├─────»
                          │               │              └────┬─────┘     »
 q_4: ────────────────────┼───────────────┼───────────────────┼───────────»
                          │               │                   │           »
 q_5: ────────────────────┼───────────────┼───────────────────┼───────────»
                          │               │ 

/opt/anaconda3/envs/qiskit-fresh/lib/python3.10/site-packages/numpy/linalg/_linalg.py:2383: RuntimeWarning: divide by zero encountered in det
  r = _umath_linalg.det(a, signature=signature)
/opt/anaconda3/envs/qiskit-fresh/lib/python3.10/site-packages/numpy/linalg/_linalg.py:2383: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


||top_left_block - expected L_total|| = 5.005e-14
||projected - L_total v|| = 4.450e-15
